# Inspect Dataset\n\nScientific sanity-check notebook for Phase 1 weak-lensing dataset generation.

## 1) Load Dataset

In [ ]:
import os\nimport h5py\nimport numpy as np\nimport matplotlib.pyplot as plt\n\nDATASET_DIR = 'datasets'\nSPLITS = ['train', 'validation', 'test']\npaths = {s: os.path.join(DATASET_DIR, s, f'dataset_{s}.h5') for s in SPLITS}\npaths

## 2) Inspect HDF5 Structure

In [ ]:
def print_structure(path):\n    print(f'\n=== {path} ===')\n    with h5py.File(path, 'r') as f:\n        def visitor(name, obj):\n            if isinstance(obj, h5py.Dataset):\n                print(f'DATASET {name}: shape={obj.shape}, dtype={obj.dtype}')\n            else:\n                print(f'GROUP   {name}')\n        f.visititems(visitor)\n        print('\nmetadata attrs:')\n        for k, v in f['metadata'].attrs.items():\n            print(f'  {k}: {v}')\n\nfor s, p in paths.items():\n    if os.path.exists(p):\n        print_structure(p)

## 3) Inspect Parameter Distributions

In [ ]:
params = {s: {} for s in SPLITS}\nfor s, p in paths.items():\n    if not os.path.exists(p):\n        continue\n    with h5py.File(p, 'r') as f:\n        for k in ['z', 'h', 'OmegaM', 'sigma8']:\n            params[s][k] = f[f'samples/{k}'][:]\n\nfig, axes = plt.subplots(2, 2, figsize=(12, 8))\nkeys = ['z', 'h', 'OmegaM', 'sigma8']\ncolors = {'train':'tab:blue', 'validation':'tab:green', 'test':'tab:red'}\nfor ax, key in zip(axes.flatten(), keys):\n    for s in SPLITS:\n        if key in params[s]:\n            ax.hist(params[s][key], bins=20, alpha=0.5, label=s, color=colors[s])\n    ax.set_title(key)\n    ax.grid(alpha=0.3)\naxes[0,0].legend()\nplt.tight_layout()

## 4) Inspect ln(mu) Distributions

In [ ]:
train_path = paths['train']\nwith h5py.File(train_path, 'r') as f:\n    lnmu = f['samples/lnmu']\n    counts = f['samples/valid_counts'][:]\n    z = f['samples/z'][:]\n\nix = np.linspace(0, len(z)-1, min(5, len(z)), dtype=int)\nplt.figure(figsize=(10, 6))\nfor i in ix:\n    n = int(counts[i])\n    row = lnmu[i, :n]\n    row = row[np.isfinite(row)]\n    if len(row):\n        plt.hist(row, bins=60, density=True, histtype='step', lw=2, label=f'z={z[i]:.2f}')\nplt.xlabel('ln(mu)')\nplt.ylabel('PDF')\nplt.title('Train ln(mu) histograms by redshift slice')\nplt.grid(alpha=0.3)\nplt.legend()\nplt.tight_layout()

In [ ]:
# Tail and redshift evolution diagnostics\nzs, vars_ = [], []\nplt.figure(figsize=(10, 6))\nfor i in ix:\n    n = int(counts[i])\n    row = np.sort(lnmu[i, :n])\n    if len(row) == 0:\n        continue\n    surv = 1.0 - (np.arange(1, len(row)+1) / len(row))\n    plt.plot(row, np.maximum(surv, 1e-6), label=f'z={z[i]:.2f}')\nplt.yscale('log')\nplt.xlabel('ln(mu)')\nplt.ylabel('1-CDF')\nplt.title('Tail survival function')\nplt.legend()\nplt.grid(alpha=0.3)\nplt.tight_layout()\n\nfor i in range(len(z)):\n    n = int(counts[i])\n    row = lnmu[i, :n]\n    row = row[np.isfinite(row)]\n    if len(row) > 1:\n        zs.append(z[i])\n        vars_.append(np.var(row))\n\nplt.figure(figsize=(8,5))\nplt.scatter(zs, vars_, s=14, alpha=0.7)\nplt.xlabel('z')\nplt.ylabel('Var[ln(mu)]')\nplt.title('Redshift evolution of variance')\nplt.grid(alpha=0.3)\nplt.tight_layout()

## 5) Inspect Train/Validation/Test Coverage

In [ ]:
keys = ['z','h','OmegaM','sigma8']\nfig, axes = plt.subplots(4, 4, figsize=(12, 12))\nfor i, ki in enumerate(keys):\n    for j, kj in enumerate(keys):\n        ax = axes[i, j]\n        if i == j:\n            for s in SPLITS:\n                if ki in params[s]:\n                    ax.hist(params[s][ki], bins=15, alpha=0.4, label=s)\n        else:\n            for s in SPLITS:\n                if ki in params[s] and kj in params[s]:\n                    ax.scatter(params[s][kj], params[s][ki], s=10, alpha=0.5, label=s if (i,j)==(0,1) else None)\n        if i == 3:\n            ax.set_xlabel(kj)\n        if j == 0:\n            ax.set_ylabel(ki)\naxes[0,1].legend(loc='best')\nplt.suptitle('Interpolation-aware split coverage')\nplt.tight_layout()

## 6) Inspect valid_counts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))\nfor s in SPLITS:\n    p = paths[s]\n    if not os.path.exists(p):\n        continue\n    with h5py.File(p, 'r') as f:\n        c = f['samples/valid_counts'][:]\n        ns = f['samples/lnmu'].shape[1]\n    vf = c / ns\n    axes[0].hist(vf, bins=20, alpha=0.5, label=s)\n    axes[1].hist(c, bins=20, alpha=0.5, label=s)\naxes[0].set_title('Valid fraction per config')\naxes[0].set_xlabel('valid_count / nsamples')\naxes[1].set_title('Valid count histogram')\naxes[1].set_xlabel('valid_count')\nfor ax in axes:\n    ax.grid(alpha=0.3)\n    ax.legend()\nplt.tight_layout()

## 7) Inspect Normalization Metadata

In [ ]:
for s, p in paths.items():\n    if not os.path.exists(p):\n        continue\n    with h5py.File(p, 'r') as f:\n        pre = f['metadata/preprocessing'].attrs\n        print(f'\n[{s}]')\n        for k in ['lnmu_mean', 'lnmu_std', 'lnmu_min', 'lnmu_max']:\n            print(f'  {k}: {pre.get(k)}')

## 8) Wasserstein Smoothness Sanity Checks

In [ ]:
def wasserstein_1d(u, v):\n    u = np.sort(u[np.isfinite(u)])\n    v = np.sort(v[np.isfinite(v)])\n    if len(u) == 0 or len(v) == 0:\n        return np.nan\n    n = min(len(u), len(v))\n    uq = np.interp(np.linspace(0,1,n), np.linspace(0,1,len(u)), u)\n    vq = np.interp(np.linspace(0,1,n), np.linspace(0,1,len(v)), v)\n    return np.mean(np.abs(uq - vq))\n\nwith h5py.File(train_path, 'r') as f:\n    lnmu = f['samples/lnmu']\n    counts = f['samples/valid_counts'][:]\n    s8 = f['samples/sigma8'][:]\n\norder = np.argsort(s8)\ndists = []\nfor i in range(len(order)-1):\n    a, b = order[i], order[i+1]\n    na, nb = int(counts[a]), int(counts[b])\n    if na == 0 or nb == 0:\n        continue\n    d = wasserstein_1d(lnmu[a,:na], lnmu[b,:nb])\n    dists.append(d)\n\nplt.figure(figsize=(8,4))\nplt.plot(dists, marker='o')\nplt.title('Adjacent-in-sigma8 Wasserstein distances (train split)')\nplt.xlabel('pair index')\nplt.ylabel('W1 distance')\nplt.grid(alpha=0.3)\nplt.tight_layout()\nprint('mean W1:', np.nanmean(dists), 'std W1:', np.nanstd(dists))